# REPO ORBIT — R3 uv/Linux repair (CPU / Colab)

This notebook reuses the **accepted Requests and Flask Linux evidence** from R2 and reruns **only uv/Linux** on a complete, non-shallow pinned Git history. It does **not** install or execute code from those repositories. All baseline and candidate Python files are verified against pinned hashes before benchmarking.

**Input:** `REPO_ORBIT_v0.8.0_EXTERNAL_VALIDATION_R3_UV_RETRY.zip` (single ZIP). **Output:** `REPO_ORBIT_COLAB_LINUX_RESULTS_R3.zip`. A Linux acceptance cannot substitute for native Windows or macOS. Running the notebook requires GitHub network access; no cloud API key or GPU is required.


In [ ]:
from google.colab import files
from pathlib import Path
import io, zipfile, shutil, json, hashlib
uploads=files.upload()
assert len(uploads)==1, 'Upload exactly one REPO ORBIT R3 ZIP file'
name,blob=next(iter(uploads.items()))
assert name=='REPO_ORBIT_v0.8.0_EXTERNAL_VALIDATION_R3_UV_RETRY.zip', f'Unexpected ZIP: {name}'
root=Path('/content/repo_orbit_native_r3')
if root.exists(): shutil.rmtree(root)
root.mkdir(parents=True)
with zipfile.ZipFile(io.BytesIO(blob)) as z:
    assert z.testzip() is None, 'Invalid ZIP member checksum'
    for entry in z.infolist():
        target=(root/entry.filename).resolve()
        if not target.is_relative_to(root.resolve()):raise ValueError('Unsafe ZIP path')
        if (entry.external_attr >> 16) & 0o170000 == 0o120000:raise ValueError('Symlink not allowed')
    z.extractall(root)
project=root/'REPO_ORBIT'
assert project.is_dir()
config=json.loads((project/'validation/PINNED_TARGETS.json').read_text())
for ver,folder in [('v0.7.0',project/'baselines/v0.7.0/repoorbit'),('v0.8.0',project/'repoorbit')]:
    for part,expected in config['source_sha256'][ver].items():
        actual=hashlib.sha256((folder/(part+'.py')).read_bytes()).hexdigest()
        assert actual==expected, f'Engine integrity FAIL {ver}/{part}'
manifest=json.loads((project/'validation/R3_ADDITIONS_SHA256.json').read_text())
for rel,digest in manifest['added_files_sha256'].items():
    assert hashlib.sha256((project/rel).read_bytes()).hexdigest()==digest,f'Addendum hash FAIL: {rel}'
results=project/'external-results/colab-linux'
for target in ['requests','flask']:
    p=next(results.glob(f'{target}_Linux_*.json'))
    record=json.loads(p.read_text())
    assert record['status']=='PASS' and record['metadata']['is_shallow'] is False
    assert record['metadata']['HEAD']==config['repositories'][target]['sha']
print('R3 package and source hashes PASS; accepted Requests/Flask evidence preserved')


In [ ]:
import subprocess,sys
preflight=project/'scripts/prepare_uv_full_history.py'
preflight_record=results/'uv_full_history_preflight.json'
cmd=[sys.executable,str(preflight),'--project',str(project),'--cache',str(project/'external-git-cache/uv'),'--out',str(preflight_record)]
p=subprocess.run(cmd,cwd=project,capture_output=True,text=True)
(results/'uv_full_history_preflight.log').write_text(p.stdout+'\n'+p.stderr)
print(p.stdout[-6000:]);print(p.stderr[-1800:] if p.stderr else '')
assert p.returncode==0, f'uv full-history acquisition failed (exit {p.returncode}). Keep evidence on HOLD.'
assert json.loads(preflight_record.read_text())['git_shallow'] is False


In [ ]:
cmd=[sys.executable,'scripts/run_native_matrix.py','--targets','uv','--offline','--runs','21','--commits','60','300','1000','--verify','sample','--out','external-results/colab-linux']
p=subprocess.run(cmd,cwd=project,capture_output=True,text=True)
print(p.stdout[-6000:]);print(p.stderr[-2500:] if p.stderr else '');print('Exit code',p.returncode)
if p.returncode==0:
    uvfile=next(results.glob('uv_Linux_*.json'))
    uv=json.loads(uvfile.read_text())
    assert uv['metadata']['is_shallow'] is False, 'uv full-history validation failed'
    assert all(c['parity']=={'canonical_json':'PASS','browser_dataset':'PASS'} for c in uv['cases'])
else:
    print('uv benchmark not accepted; export evidence and inspect logs. No forced PASS.')


In [ ]:
from datetime import datetime,timezone
outaudit=project/'external-results/colab_matrix_audit_R3.json'
cmd=[sys.executable,'scripts/audit_native_matrix.py','--inputs',str(results),'--browser-inputs',str(project/'external-results/browser_Linux.json'),'--out',str(outaudit)]
p=subprocess.run(cmd,cwd=project,capture_output=True,text=True)
print(p.stdout[-6000:]);print(p.stderr[-2500:] if p.stderr else '')
audit=json.loads(outaudit.read_text())
print('CLI performance matrix:',audit['cli_performance_gate'])
print('Valid native Linux repositories:',audit['valid_host_repo_cells'])
print('Invalid evidence:',audit['invalid_evidence'])
print('Missing cells:',audit['missing_host_repo_cells'])
print('Browser cross-platform:',audit['browser_cross_platform_gate']['status'])
# Save all Colab outputs, including any error evidence, with no fabricated PASS.
outzip=Path('/content/REPO_ORBIT_COLAB_LINUX_RESULTS_R3.zip')
with zipfile.ZipFile(outzip,'w',compression=zipfile.ZIP_DEFLATED) as z:
    selected=[f for f in results.rglob('*') if f.is_file()]
    selected += [outaudit,outaudit.with_suffix('.md'),project/'validation/PINNED_TARGETS.json',project/'validation/R3_ADDITIONS_SHA256.json']
    selected += [project/'external-results/browser_Linux.json']
    for f in selected:z.write(f,arcname=str(f.relative_to(project)))
files.download(str(outzip))
print('R3 evidence exported. A three-repository Linux PASS still leaves Windows and macOS pending.')
